# Fake.br: experimento Local Outlier Factor para desvio linguístico

Notebook independente derivado do protocolo do Isolation Forest. Os quatro detectores LOF
aprendem somente com notícias **True**; labels Fake aparecem apenas na seleção assistida por
validação e na avaliação do teste histórico. `anomalyScore` é um sinal de desvio linguístico,
não uma probabilidade de falsidade e não comprova que uma notícia seja falsa.

**Recorte implementado:** pipelines LOF fechados com `n_neighbors` 10, 20, 40 e 80,
`novelty=True`, diagnóstico do fit separado da pontuação de novidade, seleção ROC-AUC → AP →
menor `n_neighbors` na validação, corte q95 calibrado somente em True-validation, controles
Isolation Forest e One-Class SVM congelado, avaliação por ID, casos extremos e transições.


## 1. Imports e configuração do ambiente

Execute esta célula em um kernel limpo. No Colab, as bibliotecas `numpy`, `pandas`, `scikit-learn` e `matplotlib` normalmente já estão disponíveis; se o ambiente solicitar, instale-as antes de executar todas as células.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile
from itertools import combinations
from importlib.metadata import version
import hashlib
import platform
import json
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve, precision_recall_curve,
    ConfusionMatrixDisplay,
)

In [ ]:
RANDOM_STATE = 42
pd.set_option("display.max_rows", 80)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.2})
print("Python:", platform.python_version())
print({name: version(name) for name in ["numpy", "pandas", "scikit-learn", "matplotlib", "ipykernel"]})
BASE_NOTEBOOK_SHA256 = "25ff9d35c233c9bd6067ce0ebcf14bc78a077659f5c769a15b8f4486bd542d79"
OCSVM_NOTEBOOK_SHA256 = "87660b32622b7a0a4f9dfc0c0458b62983215228fe6d01e31729b6e48a2eaddb"
EXPERIMENT_KEY = "local-outlier-factor-core-confirmatory"


## 3. Carregamento dos dados

Mesmo ZIP e revisão fixa do notebook original. Textos completos e metadados são
carregados para auditoria; o truncamento acontece ANTES da extração usada nos
modelos. Nenhuma notícia é excluída por comprimento.

In [ ]:
CORPUS_REVISION = "780f5516c4ae070761632d98ac3368f3ded09d35"
CORPUS_URL = f"https://codeload.github.com/roneysco/Fake.br-Corpus/zip/{CORPUS_REVISION}"
projectFolder = Path.cwd()
dataFolder = projectFolder / "data"
dataFolder.mkdir(exist_ok=True)
archivePath = dataFolder / f"Fake.br-Corpus-{CORPUS_REVISION}.zip"

if not archivePath.exists():
    temporaryPath = archivePath.with_suffix(".download")
    with urlopen(CORPUS_URL, timeout=120) as response, temporaryPath.open("wb") as target:
        while chunk := response.read(1024 * 1024):
            target.write(chunk)
    with ZipFile(temporaryPath) as archive:
        assert archive.testzip() is None, "ZIP corrompido; refaça o download."
    temporaryPath.replace(archivePath)

print("Corpus revision:", CORPUS_REVISION)
print("Archive SHA256:", hashlib.sha256(archivePath.read_bytes()).hexdigest())

In [ ]:
metadataColumns = [
    "autor", "link", "categoria", "data_publicacao",
    "num_tokens", "num_palavras", "num_types", "num_links", "num_maiusculas",
    "num_verbos", "num_verbos_subj_imp", "num_substantivos", "num_adjetivos",
    "num_adverbios", "num_verbos_modais", "num_pron_1_2_sing", "num_pron_1_plural",
    "num_pronomes", "pausalidade", "num_caracteres", "tam_medio_sentenca",
    "tam_medio_palavra", "pct_erros_ortograficos", "emotividade", "diversidade",
]

In [ ]:
def loadNewsTexts(archive, folder, label):
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}/" not in name or not name.endswith(".txt"):
            continue
        articleId = Path(name).stem + ("t" if label == 0 else "")
        records.append({
            "id": articleId, "text": archive.read(name).decode("utf-8"),
            "label": label, "sourceClass": folder,
        })
    assert records, f"Nenhum texto encontrado em {folder}"
    return pd.DataFrame(records)


def loadNewsMetadata(archive, folder, label):
    records = []
    for name in sorted(archive.namelist()):
        if f"/full_texts/{folder}-meta-information/" not in name or not name.endswith("-meta.txt"):
            continue
        values = [line.strip() for line in archive.read(name).decode("utf-8").splitlines()]
        assert len(values) == len(metadataColumns), f"Esquema inesperado em {name}: {len(values)} linhas"
        articleId = Path(name).name.removesuffix("-meta.txt") + ("t" if label == 0 else "")
        records.append({**dict(zip(metadataColumns, values)), "id": articleId, "metadataLabel": label})
    assert records, f"Nenhum metadado encontrado em {folder}"
    return pd.DataFrame(records)

In [ ]:
with ZipFile(archivePath) as archive:
    textsFrame = pd.concat([
        loadNewsTexts(archive, "fake", 1), loadNewsTexts(archive, "true", 0),
    ], ignore_index=True)
    metadataFrame = pd.concat([
        loadNewsMetadata(archive, "fake", 1), loadNewsMetadata(archive, "true", 0),
    ], ignore_index=True)

assert textsFrame["id"].is_unique and metadataFrame["id"].is_unique
assert set(textsFrame["id"]) == set(metadataFrame["id"]), "Texto/metadados sem correspondência"
newsFrame = textsFrame.merge(metadataFrame, on="id", validate="one_to_one", indicator=True)
assert newsFrame["_merge"].eq("both").all()
assert newsFrame["label"].eq(newsFrame["metadataLabel"]).all()
newsFrame = newsFrame.drop(columns=["_merge", "metadataLabel"])
assert newsFrame["text"].str.strip().ne("").all()
print(f"{len(newsFrame):,} notícias carregadas; textos e metadados correspondem 1:1.")

## 4. Contrato dos labels

**0 = True; 1 = Fake.** O rótulo forma as partições e permite avaliação externa.
Não entra como feature. A origem nas pastas também é verificada.

In [ ]:
labelNames = {0: "True", 1: "Fake"}
assert labelNames == {0: "True", 1: "Fake"}
assert set(newsFrame["label"].unique()) == {0, 1}
assert newsFrame.loc[newsFrame["label"].eq(0), "sourceClass"].eq("true").all()
assert newsFrame.loc[newsFrame["label"].eq(1), "sourceClass"].eq("fake").all()
display(newsFrame.groupby(["label", "sourceClass"]).size().rename("newsCount").to_frame())

## 5. Preparação dos metadados

Preservamos todos os campos brutos em `rawNewsFrame` e todas as contagens numéricas
em `newsFrame`. Valores não numéricos viram NaN com contagem explícita; nenhum
ausente é preenchido antes do treino. `tem_autor` segue a lógica do original:
ausência para string vazia, `None`, `none` ou `NULL`. Não é uma medida de credibilidade.

In [ ]:
rawNewsFrame = newsFrame.copy(deep=True)
numericMetadataColumns = metadataColumns[4:]
convertedMetadata = newsFrame[numericMetadataColumns].apply(pd.to_numeric, errors="coerce")
conversionMissing = convertedMetadata.isna().sum().rename("missingAfterNumericConversion")
display(conversionMissing.to_frame())
newsFrame[numericMetadataColumns] = convertedMetadata
newsFrame["tem_autor"] = (~newsFrame["autor"].fillna("").astype(str).str.strip().isin(
    ["", "None", "none", "NULL"]
)).astype(int)

## 6. Extração no texto efetivamente utilizado

Normalização Unicode NFKC, remoção do BOM inicial e primeiros CHARACTER_LIMIT
caracteres, incluindo espaços/pontuação. Textos menores permanecem menores, sem
preenchimento. O corte pode dividir palavras/frases. Palavras são sequências de
letras com hífen/apóstrofo interno; tokens também incluem números e pontuação.
Tipos são palavras distintas ignorando caixa. TTR = tipos/tokens; diversidade =
tipos/palavras. Maiúsculas conta palavras totalmente maiúsculas com mais de uma
letra. Links são URLs http(s) presentes no corpo. Estas definições explícitas
não pretendem reproduzir o extrator desconhecido dos metadados históricos.

As demais contagens linguísticas são marcadas ausentes na cópia de features,
não imputadas nem selecionadas pelo modelo; originais ficam em rawNewsFrame e
newsFrame. Autor permanece como metadado válido da notícia inteira.

In [ ]:
def calculateRatio(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce").astype(float)
    denominator = pd.to_numeric(denominator, errors="coerce").astype(float)
    safeDenominator = denominator.where(denominator.gt(0) & np.isfinite(denominator))
    return numerator.div(safeDenominator).replace([np.inf, -np.inf], np.nan)

In [ ]:
import re
import unicodedata

CHARACTER_LIMIT = 300
numericMetadataColumns = metadataColumns[4:]
anomalyColumns = ["tem_autor", "typeTokenRatio", "linkDensity", "punctuationDensity", "uppercaseRatio", "diversidade"]
wordPattern = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*", re.UNICODE)
tokenPattern = re.compile(r"[^\W\d_]+(?:['’\-][^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]", re.UNICODE)


def measureText(text, characterLimit):
    normalized = unicodedata.normalize("NFKC", text).lstrip("\ufeff")
    if characterLimit is not None:
        if not isinstance(characterLimit, int) or characterLimit < 1:
            raise ValueError("Limite deve ser inteiro positivo ou None.")
        normalized = normalized[:characterLimit]
    words = wordPattern.findall(normalized)
    return {"text": normalized, "num_palavras": len(words),
            "num_tokens": len(tokenPattern.findall(normalized)),
            "num_types": len({word.casefold() for word in words}),
            "num_links": len(re.findall(r"https?://\S+", normalized, flags=re.IGNORECASE)),
            "num_maiusculas": sum(word.isupper() and len(word) > 1 for word in words),
            "num_caracteres": len(normalized)}


def extractAnomalyFeatures(newsFrame, characterLimit=CHARACTER_LIMIT):
    featuresFrame = newsFrame.copy(deep=True)
    featuresFrame["num_palavras_original"] = newsFrame["num_palavras"]
    featuresFrame["text_original"] = newsFrame["text"]
    featuresFrame[numericMetadataColumns] = np.nan
    measurements = pd.DataFrame([measureText(text, characterLimit) for text in newsFrame["text"]], index=newsFrame.index)
    for column in measurements:
        featuresFrame[column] = measurements[column]
    featuresFrame["typeTokenRatio"] = calculateRatio(featuresFrame["num_types"], featuresFrame["num_tokens"])
    featuresFrame["diversidade"] = calculateRatio(featuresFrame["num_types"], featuresFrame["num_palavras"])
    featuresFrame["linkDensity"] = calculateRatio(featuresFrame["num_links"], featuresFrame["num_palavras"])
    featuresFrame["punctuationDensity"] = calculateRatio(featuresFrame["num_tokens"] - featuresFrame["num_palavras"], featuresFrame["num_tokens"])
    featuresFrame["uppercaseRatio"] = calculateRatio(featuresFrame["num_maiusculas"], featuresFrame["num_palavras"])
    return featuresFrame

In [ ]:
featuresFrame = extractAnomalyFeatures(newsFrame)
featuresFrame[anomalyColumns] = featuresFrame[anomalyColumns].replace([np.inf, -np.inf], np.nan)
display(featuresFrame[anomalyColumns].isna().sum().rename("missingCount").to_frame())
print("Limite de caracteres:", CHARACTER_LIMIT)
display(featuresFrame.groupby("label")[["num_palavras_original", "num_palavras", "num_caracteres"]].agg(["min", "median", "max"]))
print("Textos menores que o limite:", int(featuresFrame["num_caracteres"].lt(CHARACTER_LIMIT).sum()))

## 7. Sanity checks

As seis features utilizam o prefixo; a presença de autor vem dos metadados.
Contagens originais ficam preservadas para auditoria e não alimentam o modelo.

In [ ]:
assert "num_palavras" not in anomalyColumns
assert "label" not in anomalyColumns and "id" not in anomalyColumns
assert len(anomalyColumns) == len(set(anomalyColumns)) == 6
assert featuresFrame["id"].is_unique
assert featuresFrame["num_caracteres"].le(CHARACTER_LIMIT).all()
assert featuresFrame["num_palavras_original"].equals(newsFrame["num_palavras"])
assert featuresFrame["num_verbos"].isna().all()
assert not np.isinf(featuresFrame[anomalyColumns].to_numpy(dtype=float)).any()
assert measureText("casa " * 100, 300)["num_palavras"] == 60
assert measureText("casa " * 100, 300)["num_caracteres"] == 300
print("Contagens do prefixo verificadas; num_palavras não entra no treinamento.")

## 8. Análise estatística

Estatísticas descritivas por label, sem substituir vetores individuais por médias.
Esta inspeção do corpus completo atende ao protocolo exploratório: não usamos
seus resultados para selecionar features, ajustar hiperparâmetros ou threshold.
Qualquer escolha futura guiada por estes resultados exige nova avaliação independente.
Correlações de Pearson com comprimento são mostradas no total e por classe para
evitar confundir efeitos de classe e de tamanho. NaN em correlação pode indicar
feature constante. Mantemos as seis features recalculadas, inclusive possíveis redundâncias.

In [ ]:
statisticsRecords = []
for label, group in featuresFrame.groupby("label", sort=True):
    for feature in anomalyColumns:
        values = group[feature]
        q1, q3 = values.quantile([0.25, 0.75])
        statisticsRecords.append({
            "label": label, "class": labelNames[label], "feature": feature,
            "count": values.count(), "mean": values.mean(), "median": values.median(),
            "std": values.std(), "min": values.min(), "Q1": q1, "Q3": q3,
            "IQR": q3 - q1, "max": values.max(), "missingCount": values.isna().sum(),
        })
featureStatistics = pd.DataFrame(statisticsRecords).set_index(["label", "class", "feature"])
display(featureStatistics)

lengthColumns = ["num_palavras", "num_tokens"]
lengthCorrelations = pd.concat({
    name: group[anomalyColumns + lengthColumns].corr().loc[anomalyColumns, lengthColumns]
    for name, group in [
        ("All", featuresFrame),
        ("True (0)", featuresFrame.loc[featuresFrame["label"].eq(0)]),
        ("Fake (1)", featuresFrame.loc[featuresFrame["label"].eq(1)]),
    ]
}, names=["group", "feature"])
display(lengthCorrelations)
featureCorrelations = featuresFrame[anomalyColumns].corr()
display(featureCorrelations.round(3))
display(lengthCorrelations.xs("typeTokenRatio", level="feature"))
print("Pearson TTR vs diversidade:", featureCorrelations.loc["typeTokenRatio", "diversidade"])

## 9. Train / Validation / Test

True: 60% treino, 20% validação, 20% teste. Fake: 50% validação, 50% teste.
Todas as divisões usam `random_state=42`. Verificamos IDs exclusivos e cobertura
integral do corpus. **Nenhuma notícia Fake participa de fit ou calibração.**

Limitação do protocolo solicitado: a divisão é por notícia, não por assunto,
fonte ou data. O corpus possui pares True/Fake com o mesmo número-base; IDs
`123t` e `123` são notícias distintas, mas podem tratar do mesmo assunto em
partições diferentes. IDs exclusivos não demonstram independência temática.

In [ ]:
normalFrame = featuresFrame.loc[featuresFrame["label"].eq(0)].copy()
fakeFrame = featuresFrame.loc[featuresFrame["label"].eq(1)].copy()
normalTrainFrame, normalHoldoutFrame = train_test_split(
    normalFrame, train_size=0.60, random_state=RANDOM_STATE,
)
normalValidationFrame, normalTestFrame = train_test_split(
    normalHoldoutFrame, test_size=0.50, random_state=RANDOM_STATE,
)
fakeValidationFrame, fakeTestFrame = train_test_split(
    fakeFrame, test_size=0.50, random_state=RANDOM_STATE,
)
partitions = {
    "normalTrain": normalTrainFrame, "normalValidation": normalValidationFrame,
    "normalTest": normalTestFrame, "fakeValidation": fakeValidationFrame,
    "fakeTest": fakeTestFrame,
}
for name, frame in partitions.items():
    assert not frame.empty and frame["id"].is_unique
    assert frame["label"].eq(0 if name.startswith("normal") else 1).all()
for (leftName, left), (rightName, right) in combinations(partitions.items(), 2):
    assert set(left["id"]).isdisjoint(right["id"]), f"IDs compartilhados: {leftName}/{rightName}"
assert set().union(*(set(frame["id"]) for frame in partitions.values())) == set(featuresFrame["id"])
assert sum(len(frame) for frame in partitions.values()) == len(featuresFrame)
assert normalTrainFrame["label"].eq(0).all()
display(pd.DataFrame([
    {"partition": name, "count": len(frame), "label": int(frame["label"].iloc[0])}
    for name, frame in partitions.items()
]).set_index("partition"))

## 10. Local Outlier Factor, novidade e guardas True-only

Cada candidato tem imputer, `StandardScaler` e LOF próprios. O ajuste recebe somente
`normalTrainFrame`. O LOF usa `novelty=True`, `contamination="auto"`, distância Minkowski
com `p=2` e `n_jobs=1`. As APIs de novidade nunca são chamadas nas próprias amostras usadas
no fit; o treino é descrito somente por `negative_outlier_factor_`.


In [ ]:
LOF_CANDIDATES = {
    "lof_neighbors_010": 10,
    "lof_neighbors_020": 20,
    "lof_neighbors_040": 40,
    "lof_neighbors_080": 80,
}


def buildLofPipeline(nNeighbors):
    if nNeighbors not in LOF_CANDIDATES.values():
        raise ValueError(f"n_neighbors fora da matriz pré-registrada: {nNeighbors}")
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "detector",
            LocalOutlierFactor(
                n_neighbors=nNeighbors,
                algorithm="auto",
                metric="minkowski",
                p=2,
                contamination="auto",
                novelty=True,
                n_jobs=1,
            ),
        ),
    ])


def buildIsolationForestControlPipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("detector", IsolationForest(
            n_estimators=300,
            contamination="auto",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])


def buildOneClassSvmControlPipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("detector", OneClassSVM(kernel="rbf", gamma="scale", nu=0.10)),
    ])


def fitNormalOnly(pipeline, trainingFrame):
    if trainingFrame.empty or not trainingFrame["label"].eq(0).all():
        raise ValueError("Treino permitido apenas com notícias True (label == 0).")
    trainingFeatures = trainingFrame[anomalyColumns]
    if np.isinf(trainingFeatures.to_numpy(dtype=float)).any():
        raise ValueError("Features de treino contêm infinito.")
    emptyColumns = trainingFeatures.columns[trainingFeatures.isna().all()].tolist()
    if emptyColumns:
        raise ValueError(f"Features inteiramente ausentes em normalTrain: {emptyColumns}")
    fitted = pipeline.fit(trainingFeatures)
    fitted._normalTrainIds = frozenset(trainingFrame["id"].astype(str))
    return fitted


def _assertNoveltyFrame(pipeline, frame):
    fitIds = getattr(pipeline, "_normalTrainIds", frozenset())
    if fitIds and "id" in frame.columns:
        overlap = fitIds.intersection(frame["id"].astype(str))
        if overlap:
            raise AssertionError(
                "APIs de novidade não podem pontuar amostras usadas no fit; "
                f"IDs sobrepostos: {len(overlap)}"
            )


def anomalyScores(pipeline, frame):
    _assertNoveltyFrame(pipeline, frame)
    scores = -pipeline.decision_function(frame[anomalyColumns])
    if not np.isfinite(scores).all():
        raise ValueError("Scores não finitos.")
    return np.asarray(scores, dtype=float)


def nativeFlags(pipeline, frame):
    _assertNoveltyFrame(pipeline, frame)
    normalityScore = pipeline.decision_function(frame[anomalyColumns])
    decisionFlags = normalityScore < 0
    predictionFlags = pipeline.predict(frame[anomalyColumns]) == -1
    if not np.array_equal(predictionFlags, decisionFlags):
        raise AssertionError("Fronteira nativa divergiu de predict == -1.")
    return np.asarray(predictionFlags, dtype=bool)


def fitDiagnostics(pipeline):
    detector = pipeline.named_steps["detector"]
    if not isinstance(detector, LocalOutlierFactor):
        raise TypeError("Diagnóstico negative_outlier_factor_ exige um pipeline LOF.")
    negativeFactor = np.asarray(detector.negative_outlier_factor_, dtype=float)
    fitLofFactor = -negativeFactor

    def stats(values):
        values = np.asarray(values, dtype=float)
        quantiles = np.quantile(values, [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
        return {
            "count": int(values.size),
            "mean": float(values.mean()),
            "std": float(values.std()),
            "min": float(values.min()),
            "q01": float(quantiles[0]),
            "q05": float(quantiles[1]),
            "q25": float(quantiles[2]),
            "median": float(quantiles[3]),
            "q75": float(quantiles[4]),
            "q95": float(quantiles[5]),
            "q99": float(quantiles[6]),
            "max": float(values.max()),
        }

    return {
        "offset": float(detector.offset_),
        "nNeighbors": int(detector.n_neighbors_),
        "negativeOutlierFactor": stats(negativeFactor),
        "fitLofFactor": stats(fitLofFactor),
        "fitBelowOffsetCount": int(np.sum(negativeFactor < detector.offset_)),
        "fitBelowOffsetFraction": float(np.mean(negativeFactor < detector.offset_)),
    }


def binaryMetrics(labels, scores, flags):
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=float)
    flags = np.asarray(flags, dtype=bool)
    matrix = confusion_matrix(labels, flags.astype(int), labels=[0, 1])
    tn, fp, fn, tp = (int(value) for value in matrix.ravel())
    return {
        "rocAuc": float(roc_auc_score(labels, scores)),
        "averagePrecision": float(average_precision_score(labels, scores)),
        "fakePrevalence": float(np.mean(labels == 1)),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "accuracy": float((tp + tn) / matrix.sum()),
        "precision": float(precision_score(labels, flags, zero_division=0)),
        "recall": float(recall_score(labels, flags, zero_division=0)),
        "f1": float(f1_score(labels, flags, zero_division=0)),
        "fpr": float(fp / (tn + fp)) if tn + fp else 0.0,
        "confusionMatrix": matrix.astype(int).tolist(),
    }


def selectCandidate(validationRecords, tolerance=1e-12):
    if not validationRecords:
        raise ValueError("A seleção exige resultados de validação.")
    bestAuc = max(record["validationRocAuc"] for record in validationRecords)
    aucTies = [
        record for record in validationRecords
        if bestAuc - record["validationRocAuc"] <= tolerance
    ]
    bestAp = max(record["validationAveragePrecision"] for record in aucTies)
    apTies = [
        record for record in aucTies
        if bestAp - record["validationAveragePrecision"] <= tolerance
    ]
    return min(apTies, key=lambda record: record["n_neighbors"])["modelKey"]


def _pairTransition(left, right, leftName, rightName):
    if left and right:
        return "alerta_mantido"
    if not left and not right:
        return "sem_alerta"
    if left and not right:
        return f"alerta_adicionado_{leftName}"
    return f"alerta_adicionado_{rightName}"


def jsonReady(value):
    if isinstance(value, dict):
        return {str(key): jsonReady(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [jsonReady(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if pd.isna(value) if value is None else False:
        return None
    return value


def writeJson(path, value):
    path.write_text(
        json.dumps(jsonReady(value), indent=2, ensure_ascii=False, allow_nan=False),
        encoding="utf-8",
    )


def rankDiscordances(predictions, controlScoreColumn):
    ranked = predictions.copy()
    ranked["lofRank"] = ranked["lofAnomalyScore"].rank(method="average", ascending=False)
    ranked["controlRank"] = ranked[controlScoreColumn].rank(method="average", ascending=False)
    ranked["absoluteRankDifference"] = (ranked["lofRank"] - ranked["controlRank"]).abs()
    columns = [
        "id", "label", "lofAnomalyScore", controlScoreColumn,
        "lofQ95IsAnomaly", "lofNativeIsAnomaly",
        "absoluteRankDifference", "lofRank", "controlRank",
    ] + anomalyColumns
    return ranked.nlargest(5, "absoluteRankDifference")[columns].to_dict(orient="records")


## 11. Treino, diagnóstico, validação e seleção congelada

Os quatro pipelines são ajustados em 2.160 True. O diagnóstico do treino usa apenas
`negative_outlier_factor_`. Para validação, `decision_function` e `predict` recebem dados
novos; o q95 é o percentil 95 dos scores das 720 True de validação. Fake-validation participa
somente da seleção assistida por ROC-AUC, AP e menor `n_neighbors`.


In [ ]:
validationFrame = pd.concat([
    normalValidationFrame.assign(partition="normalValidation"),
    fakeValidationFrame.assign(partition="fakeValidation"),
], ignore_index=True)
validationLabels = validationFrame["label"].to_numpy(dtype=int)
candidatePipelines = {}
validationRecords = []

for modelKey, nNeighbors in LOF_CANDIDATES.items():
    pipeline = buildLofPipeline(nNeighbors)
    fitStarted = time.perf_counter()
    fitNormalOnly(pipeline, normalTrainFrame)
    fitSeconds = time.perf_counter() - fitStarted

    scoreStarted = time.perf_counter()
    normalValidationScores = anomalyScores(pipeline, normalValidationFrame)
    validationScores = anomalyScores(pipeline, validationFrame)
    validationNativeFlags = nativeFlags(pipeline, validationFrame)
    scoreSeconds = time.perf_counter() - scoreStarted

    q95Threshold = float(np.quantile(normalValidationScores, 0.95))
    validationQ95Flags = validationScores >= q95Threshold
    native = binaryMetrics(validationLabels, validationScores, validationNativeFlags)
    q95 = binaryMetrics(validationLabels, validationScores, validationQ95Flags)
    detector = pipeline["detector"]
    fitInfo = fitDiagnostics(pipeline)
    record = {
        "modelKey": modelKey,
        "n_neighbors": int(nNeighbors),
        "n_neighbors_": int(detector.n_neighbors_),
        "algorithm": detector.algorithm,
        "metric": detector.metric,
        "p": int(detector.p),
        "contamination": detector.contamination,
        "novelty": bool(detector.novelty),
        "offset": float(detector.offset_),
        "offset_": float(detector.offset_),
        "normalTrainCount": int(len(normalTrainFrame)),
        "normalValidationCount": int(len(normalValidationFrame)),
        "fakeValidationCount": int(len(fakeValidationFrame)),
        "partitionCounts": {name: int(len(frame)) for name, frame in partitions.items()},
        "imputerMedian": pipeline["imputer"].statistics_.astype(float).tolist(),
        "scalerMean": pipeline["scaler"].mean_.astype(float).tolist(),
        "scalerScale": pipeline["scaler"].scale_.astype(float).tolist(),
        "fitDiagnostics": fitInfo,
        "negative_outlier_factor_": fitInfo["negativeOutlierFactor"],
        "fitLofFactor": fitInfo["fitLofFactor"],
        "fitSeconds": float(fitSeconds),
        "validationScoreSeconds": float(scoreSeconds),
        "q95Threshold": q95Threshold,
        "q95Source": "normalValidation anomalyScore only",
        "normalValidationQ95AlertRate": float(np.mean(normalValidationScores >= q95Threshold)),
        "normalValidationQ95Fpr": float(np.mean(normalValidationScores >= q95Threshold)),
        "validationRocAuc": q95["rocAuc"],
        "validationAveragePrecision": q95["averagePrecision"],
        "nativeMetrics": native,
        "q95Metrics": q95,
    }
    if record["n_neighbors_"] != nNeighbors:
        raise AssertionError("scikit-learn reduziu n_neighbors silenciosamente.")
    np.testing.assert_allclose(
        pipeline["imputer"].statistics_,
        normalTrainFrame[anomalyColumns].median().to_numpy(),
    )
    imputedTrain = pipeline["imputer"].transform(normalTrainFrame[anomalyColumns])
    np.testing.assert_allclose(pipeline["scaler"].mean_, imputedTrain.mean(axis=0))
    expectedScale = imputedTrain.std(axis=0)
    expectedScale[expectedScale == 0] = 1.0
    np.testing.assert_allclose(pipeline["scaler"].scale_, expectedScale)
    candidatePipelines[modelKey] = pipeline
    validationRecords.append(record)

selectedModelKey = selectCandidate(validationRecords)
selectedPipeline = candidatePipelines[selectedModelKey]
selectedValidationRecord = next(
    record for record in validationRecords if record["modelKey"] == selectedModelKey
)
selectedQ95Threshold = selectedValidationRecord["q95Threshold"]
validationResultsFrame = pd.DataFrame(validationRecords)
display(validationResultsFrame[[
    "modelKey", "n_neighbors", "n_neighbors_", "offset", "q95Threshold",
    "normalValidationQ95Fpr", "validationRocAuc", "validationAveragePrecision",
]])
print("Candidato LOF congelado antes do teste:", selectedModelKey)


## 12. Controles congelados e avaliação única no teste

Isolation Forest e One-Class SVM usam exatamente o mesmo `normalTrainFrame`, features,
partições e regra q95. O OCSVM é o candidato congelado `nu=0.10`; nenhum controle é
retunado no teste. Os resultados nativos e q95 ficam em tabelas separadas.


In [ ]:
isolationForestPipeline = buildIsolationForestControlPipeline()
fitNormalOnly(isolationForestPipeline, normalTrainFrame)
isolationForestNormalValidationScores = anomalyScores(
    isolationForestPipeline, normalValidationFrame,
)
isolationForestQ95Threshold = float(np.quantile(isolationForestNormalValidationScores, 0.95))

ocsvmPipeline = buildOneClassSvmControlPipeline()
fitNormalOnly(ocsvmPipeline, normalTrainFrame)
ocsvmNormalValidationScores = anomalyScores(ocsvmPipeline, normalValidationFrame)
ocsvmQ95Threshold = float(np.quantile(ocsvmNormalValidationScores, 0.95))

testFrame = pd.concat([
    normalTestFrame.assign(partition="normalTest"),
    fakeTestFrame.assign(partition="fakeTest"),
], ignore_index=True)
testLabels = testFrame["label"].to_numpy(dtype=int)

lofScores = anomalyScores(selectedPipeline, testFrame)
lofNativeFlags = nativeFlags(selectedPipeline, testFrame)
lofQ95Flags = lofScores >= selectedQ95Threshold

isolationForestScores = anomalyScores(isolationForestPipeline, testFrame)
isolationForestNativeFlags = nativeFlags(isolationForestPipeline, testFrame)
isolationForestQ95Flags = isolationForestScores >= isolationForestQ95Threshold

ocsvmScores = anomalyScores(ocsvmPipeline, testFrame)
ocsvmNativeFlags = nativeFlags(ocsvmPipeline, testFrame)
ocsvmQ95Flags = ocsvmScores >= ocsvmQ95Threshold

lofQ95Metrics = binaryMetrics(testLabels, lofScores, lofQ95Flags)
lofNativeMetrics = binaryMetrics(testLabels, lofScores, lofNativeFlags)
isolationForestQ95Metrics = binaryMetrics(
    testLabels, isolationForestScores, isolationForestQ95Flags,
)
isolationForestNativeMetrics = binaryMetrics(
    testLabels, isolationForestScores, isolationForestNativeFlags,
)
ocsvmQ95Metrics = binaryMetrics(testLabels, ocsvmScores, ocsvmQ95Flags)
ocsvmNativeMetrics = binaryMetrics(testLabels, ocsvmScores, ocsvmNativeFlags)

testPredictionsFrame = testFrame[["id", "label", "partition"] + anomalyColumns].copy()
testPredictionsFrame["lofAnomalyScore"] = lofScores
testPredictionsFrame["lofNativeIsAnomaly"] = lofNativeFlags
testPredictionsFrame["lofQ95IsAnomaly"] = lofQ95Flags
testPredictionsFrame["isolationForestAnomalyScore"] = isolationForestScores
testPredictionsFrame["isolationForestNativeIsAnomaly"] = isolationForestNativeFlags
testPredictionsFrame["isolationForestQ95IsAnomaly"] = isolationForestQ95Flags
testPredictionsFrame["ocsvmAnomalyScore"] = ocsvmScores
testPredictionsFrame["ocsvmNativeIsAnomaly"] = ocsvmNativeFlags
testPredictionsFrame["ocsvmQ95IsAnomaly"] = ocsvmQ95Flags
assert testPredictionsFrame["id"].is_unique
assert len(testPredictionsFrame) == 2520

testPredictionsFrame["lofVsIsolationForest"] = [
    _pairTransition(left, right, "lof", "isolationForest")
    for left, right in zip(lofQ95Flags, isolationForestQ95Flags)
]
testPredictionsFrame["lofVsOcsvm"] = [
    _pairTransition(left, right, "lof", "ocsvm")
    for left, right in zip(lofQ95Flags, ocsvmQ95Flags)
]
testPredictionsFrame["isolationForestVsOcsvm"] = [
    _pairTransition(left, right, "isolationForest", "ocsvm")
    for left, right in zip(isolationForestQ95Flags, ocsvmQ95Flags)
]

transitionColumns = [
    "id", "label", "partition", "lofQ95IsAnomaly",
    "isolationForestQ95IsAnomaly", "ocsvmQ95IsAnomaly",
    "lofVsIsolationForest", "lofVsOcsvm", "isolationForestVsOcsvm",
]
transitionsFrame = testPredictionsFrame[transitionColumns].copy()
transitionCounts = (
    transitionsFrame.groupby(
        ["label", "lofVsIsolationForest", "lofVsOcsvm", "isolationForestVsOcsvm"],
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
)
assert int(transitionCounts["count"].sum()) == len(testPredictionsFrame)

scoreDistributions = {}
for modelName, scoreColumn in [
    ("lof", "lofAnomalyScore"),
    ("isolationForest", "isolationForestAnomalyScore"),
    ("ocsvm", "ocsvmAnomalyScore"),
]:
    grouped = testPredictionsFrame.groupby("label")[scoreColumn].agg(
        ["count", "mean", "median", "std", "min", "max"]
    )
    scoreDistributions[modelName] = {
        str(label): row.to_dict() for label, row in grouped.iterrows()
    }

exampleColumns = [
    "id", "label", "partition", "lofAnomalyScore", "lofNativeIsAnomaly",
    "lofQ95IsAnomaly",
] + anomalyColumns
trueMostAnomalous = testPredictionsFrame.loc[
    testPredictionsFrame["label"].eq(0)
].nlargest(5, "lofAnomalyScore")[exampleColumns].to_dict(orient="records")
fakeMostAnomalous = testPredictionsFrame.loc[
    testPredictionsFrame["label"].eq(1)
].nlargest(5, "lofAnomalyScore")[exampleColumns].to_dict(orient="records")
fakeLeastAnomalous = testPredictionsFrame.loc[
    testPredictionsFrame["label"].eq(1)
].nsmallest(5, "lofAnomalyScore")[exampleColumns].to_dict(orient="records")
rankDiscordanceResults = {
    "lofVsIsolationForest": rankDiscordances(
        testPredictionsFrame, "isolationForestAnomalyScore",
    ),
    "lofVsOcsvm": rankDiscordances(testPredictionsFrame, "ocsvmAnomalyScore"),
}

testResults = {
    "selectedModelKey": selectedModelKey,
    "fakePrevalenceApBaseline": float(np.mean(testLabels == 1)),
    "thresholds": {
        "lofQ95": {
            "value": selectedQ95Threshold,
            "source": "normalValidation anomalyScore only",
            "trueValidationFpr": selectedValidationRecord["normalValidationQ95Fpr"],
        },
        "isolationForestQ95": {
            "value": isolationForestQ95Threshold,
            "source": "normalValidation anomalyScore only",
            "trueValidationFpr": float(np.mean(isolationForestNormalValidationScores >= isolationForestQ95Threshold)),
        },
        "ocsvmQ95": {
            "value": ocsvmQ95Threshold,
            "source": "normalValidation anomalyScore only",
            "trueValidationFpr": float(np.mean(ocsvmNormalValidationScores >= ocsvmQ95Threshold)),
        },
    },
    "lofQ95": lofQ95Metrics,
    "lofNative": lofNativeMetrics,
    "isolationForestQ95": isolationForestQ95Metrics,
    "isolationForestNative": isolationForestNativeMetrics,
    "ocsvmQ95": ocsvmQ95Metrics,
    "ocsvmNative": ocsvmNativeMetrics,
    "scoreDistributions": scoreDistributions,
    "transitions": transitionCounts.to_dict(orient="records"),
    "extremes": {
        "trueMostAnomalous": trueMostAnomalous,
        "fakeMostAnomalous": fakeMostAnomalous,
        "fakeLeastAnomalous": fakeLeastAnomalous,
        "rankDiscordances": rankDiscordanceResults,
    },
}

display(pd.DataFrame({
    "LOF q95": lofQ95Metrics,
    "LOF nativo": lofNativeMetrics,
    "Isolation Forest q95": isolationForestQ95Metrics,
    "Isolation Forest nativo": isolationForestNativeMetrics,
    "OCSVM q95": ocsvmQ95Metrics,
    "OCSVM nativo": ocsvmNativeMetrics,
}).T)
display(pd.DataFrame(scoreDistributions))
display(transitionCounts)


## 13. Visualizações, casos extremos e exportação

As figuras são diagnósticas: não são importância de feature nem explicação causal. Os casos
extremos e discordâncias abaixo são calculados somente depois de congelar o candidato LOF e
usam o teste histórico exclusivamente como avaliação exploratória.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10), constrained_layout=True)
validationResultsFrame.plot(
    x="n_neighbors", y="validationRocAuc", kind="bar", ax=axes[0, 0],
    title="Validação: ROC-AUC por n_neighbors", legend=False, color="tab:blue",
)
validationResultsFrame.plot(
    x="n_neighbors", y="validationAveragePrecision", kind="bar", ax=axes[0, 1],
    title="Validação: AP por n_neighbors", legend=False, color="tab:orange",
)
validationResultsFrame["fitLofFactorQ95"] = validationResultsFrame["fitDiagnostics"].map(
    lambda diagnostics: diagnostics["fitLofFactor"]["q95"]
)
validationResultsFrame.plot(
    x="n_neighbors", y="fitLofFactorQ95", kind="bar", ax=axes[0, 2],
    title="Diagnóstico do fit: q95 do fator LOF", legend=False, color="tab:green",
)

selectedTrueScores = testPredictionsFrame.loc[
    testPredictionsFrame["label"].eq(0), "lofAnomalyScore"
]
selectedFakeScores = testPredictionsFrame.loc[
    testPredictionsFrame["label"].eq(1), "lofAnomalyScore"
]
scoreBins = np.histogram_bin_edges(
    np.concatenate([selectedTrueScores, selectedFakeScores]), bins=40,
)
for values, name, color in [
    (selectedTrueScores, "True (0)", "tab:blue"),
    (selectedFakeScores, "Fake (1)", "tab:orange"),
]:
    axes[1, 0].hist(values, bins=scoreBins, density=True, alpha=0.5, label=name, color=color)
axes[1, 0].axvline(0, color="black", linestyle=":", label="LOF nativo")
axes[1, 0].axvline(selectedQ95Threshold, color="black", linestyle="--", label="q95")
axes[1, 0].set(title="LOF selecionado no teste", xlabel="anomalyScore", ylabel="Densidade")
axes[1, 0].legend(fontsize=8)

for scoreColumn, name, color in [
    ("lofAnomalyScore", "LOF", "tab:blue"),
    ("isolationForestAnomalyScore", "Isolation Forest", "tab:green"),
    ("ocsvmAnomalyScore", "OCSVM", "tab:orange"),
]:
    curveLabels = testPredictionsFrame["label"].to_numpy(dtype=int)
    curveScores = testPredictionsFrame[scoreColumn].to_numpy(dtype=float)
    curveFpr, curveTpr, _ = roc_curve(curveLabels, curveScores)
    axes[1, 1].plot(
        curveFpr, curveTpr,
        label=f"{name} AUC={roc_auc_score(curveLabels, curveScores):.3f}",
        color=color,
    )
axes[1, 1].plot([0, 1], [0, 1], "k--", label="Aleatório")
axes[1, 1].set(title="Teste: curvas ROC", xlabel="FPR", ylabel="TPR")
axes[1, 1].legend(fontsize=8)

transitionPlot = transitionsFrame.assign(
    modelPair=transitionsFrame["lofVsIsolationForest"]
).groupby("modelPair").size()
transitionPlot.plot(kind="bar", ax=axes[1, 2], color="tab:purple", title="Transições LOF × Isolation Forest")
axes[1, 2].set_xlabel("Transição q95")
axes[1, 2].set_ylabel("Notícias")
plt.show()

for title, records in [
    ("5 True mais anômalas", trueMostAnomalous),
    ("5 Fake mais anômalas", fakeMostAnomalous),
    ("5 Fake menos anômalas", fakeLeastAnomalous),
]:
    display(Markdown(f"**{title}**"))
    display(pd.DataFrame(records))
for title, records in rankDiscordanceResults.items():
    display(Markdown(f"**{title}: maiores discordâncias de rank**"))
    display(pd.DataFrame(records))


## 14. Conclusão calculada e artefatos

Quando `LOCAL_OUTLIER_FACTOR_OUTPUT_DIR` existe, esta célula salva JSONs estritos, CSVs por
ID e um relatório textual na pasta nova do run. Sem a variável, o notebook permanece
interativo e não grava evidências experimentais. O notebook-fonte nunca recebe outputs.


In [ ]:
manifest = {
    "experimentKey": EXPERIMENT_KEY,
    "baseNotebookSha256": BASE_NOTEBOOK_SHA256,
    "ocsvmNotebookSha256": OCSVM_NOTEBOOK_SHA256,
    "sourceNotebookSha256": None,
    "corpusRevision": CORPUS_REVISION,
    "archiveSha256": hashlib.sha256(archivePath.read_bytes()).hexdigest(),
    "seed": RANDOM_STATE,
    "labels": {"0": "True", "1": "Fake"},
    "characterLimit": CHARACTER_LIMIT,
    "normalization": ["NFKC", "remove-leading-BOM"],
    "features": anomalyColumns,
    "num_palavrasRole": "extraction-and-audit-only",
    "partitionCounts": {name: int(len(frame)) for name, frame in partitions.items()},
    "candidates": {
        key: {
            "n_neighbors": int(value), "algorithm": "auto", "metric": "minkowski",
            "p": 2, "contamination": "auto", "novelty": True, "n_jobs": 1,
        }
        for key, value in LOF_CANDIDATES.items()
    },
    "selectedModelKey": selectedModelKey,
    "selectionRule": "validation ROC-AUC, then AP within 1e-12, then lower n_neighbors",
    "selectionUsesFakeValidationLabels": True,
    "testWasNotUsedForSelection": True,
    "q95Source": "normalValidation anomalyScore only",
    "controls": {
        "isolationForest": {
            "n_estimators": 300, "contamination": "auto", "random_state": RANDOM_STATE,
            "n_jobs": -1, "standardScaler": False,
        },
        "oneClassSvm": {"kernel": "rbf", "gamma": "scale", "nu": 0.10},
    },
    "colabStatus": "não testado no Colab",
}
sourceNotebookPath = projectFolder / "anomaly-detection-local-outlier-factor.ipynb"
if sourceNotebookPath.exists():
    manifest["sourceNotebookSha256"] = hashlib.sha256(sourceNotebookPath.read_bytes()).hexdigest()

outputDirectory = os.environ.get("LOCAL_OUTLIER_FACTOR_OUTPUT_DIR")
runId = os.environ.get("LOCAL_OUTLIER_FACTOR_RUN_ID")
if outputDirectory:
    outputPath = Path(outputDirectory)
    outputPath.mkdir(parents=True, exist_ok=False) if not outputPath.exists() else None
    manifest["runId"] = runId
    writeJson(outputPath / "manifest.json", manifest)
    writeJson(outputPath / "validation-results.json", validationRecords)
    writeJson(outputPath / "test-results.json", testResults)
    testPredictionsFrame.to_csv(outputPath / "test-predictions.csv", index=False)
    transitionsFrame.to_csv(outputPath / "transitions.csv", index=False)
    report = f"""# Resultado do run Local Outlier Factor

Run: `{runId}`

- Candidato congelado: `{selectedModelKey}`.
- Validação ROC-AUC: `{selectedValidationRecord['validationRocAuc']:.12f}`.
- Validação AP: `{selectedValidationRecord['validationAveragePrecision']:.12f}`.
- Teste LOF q95 ROC-AUC/AP: `{lofQ95Metrics['rocAuc']:.12f}` / `{lofQ95Metrics['averagePrecision']:.12f}`.
- Teste LOF q95 TP/FP/TN/FN: `{lofQ95Metrics['tp']}/{lofQ95Metrics['fp']}/{lofQ95Metrics['tn']}/{lofQ95Metrics['fn']}`.
- Teste LOF nativo TP/FP/TN/FN: `{lofNativeMetrics['tp']}/{lofNativeMetrics['fp']}/{lofNativeMetrics['tn']}/{lofNativeMetrics['fn']}`.
- Isolation Forest q95 TP/FP/TN/FN: `{isolationForestQ95Metrics['tp']}/{isolationForestQ95Metrics['fp']}/{isolationForestQ95Metrics['tn']}/{isolationForestQ95Metrics['fn']}`.
- OCSVM q95 TP/FP/TN/FN: `{ocsvmQ95Metrics['tp']}/{ocsvmQ95Metrics['fp']}/{ocsvmQ95Metrics['tn']}/{ocsvmQ95Metrics['fn']}`.

Os scores são sinais de desvio linguístico, não probabilidades de falsidade. A seleção foi
assistida por Fake-validation; o teste é histórico e já foi consultado; o texto foi truncado em
300 caracteres; o split não é temático. **não testado no Colab**.
"""
    (outputPath / "run-report.md").write_text(report, encoding="utf-8")

display(Markdown(f"""
O candidato congelado foi **{selectedModelKey}**, com ROC-AUC de validação
**{selectedValidationRecord['validationRocAuc']:.4f}** e AP **{selectedValidationRecord['validationAveragePrecision']:.4f}**.
No teste histórico, LOF q95 obteve ROC-AUC **{lofQ95Metrics['rocAuc']:.4f}**, AP
**{lofQ95Metrics['averagePrecision']:.4f}**, recall **{lofQ95Metrics['recall']:.2%}** e FPR
**{lofQ95Metrics['fpr']:.2%}**. Estes números descrevem desvio linguístico neste split;
não provam falsidade nem superioridade estatística.

O ajuste usou **{len(normalTrainFrame)} True e nenhuma Fake**, com até **{CHARACTER_LIMIT}**
caracteres e seis features. A seleção usou labels Fake somente na validação. O teste histórico
já foi consultado, o split não é temático e **não testado no Colab**.
"""))
